In [ ]:
# ======================================================
# Notebook: Bayesian Optimisation for Drug Combination
# Goal: minimise adverse reactions (maximise -side_effects)
# ======================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import norm

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (N,3)
y_raw = np.load("/mnt/data/initial_outputs.npy") # (N,)

# Transform to maximisation
y = -y_raw

# Gaussian Process surrogate
kernel = ConstantKernel(1.0) * Matern(nu=2.5) + WhiteKernel()
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=True)
gp.fit(X, y)

# Expected Improvement
def expected_improvement(X_cand, X_sample, Y_sample, model, xi=0.01):
    mu, sigma = model.predict(X_cand, return_std=True)
    mu_sample = model.predict(X_sample)
    best = np.max(mu_sample)

    sigma = sigma.reshape(-1,1)
    mu = mu.reshape(-1,1)

    improvement = mu - best - xi
    Z = improvement / sigma
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma == 0.0] = 0.0
    return ei.ravel()

# Candidate search space (within observed bounds)
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(3)]
grid = [np.linspace(b[0], b[1], 20) for b in bounds]
X_grid = np.array(np.meshgrid(*grid)).T.reshape(-1,3)

# Acquisition
ei = expected_improvement(X_grid, X, y, gp)

# Next (10,3) experiments
top_idx = np.argsort(ei)[-10:]
next_points = X_grid[top_idx]

print("Next (10,3) compound combinations:")
print(next_points)